In [8]:
import ollama

response = ollama.chat(model='llama3', messages=[
    {'role': 'system', 'content': 'Dir werden grobe Gedanken gegeben, antworte mit 5 fertigen Posts für Social Media in deinem eigenen Stil.'},
    {'role': 'user', 'content': 'Rezepte für gesunde Ernährung'},

])
print(response.message.content)

Here are five social media post ideas on healthy recipes:

**Post 1:**
Good morning! Start your day off right with a nutritious breakfast that'll keep you going all morning! Try our Avocado Toast recipe, featuring mashed avocado, cherry tomatoes, and a sprinkle of feta cheese. Recipe link in bio! #healthybreakfast #avocadotoast #breakfastgoals

**Post 2:**
Who says dinner can't be quick AND delicious?! Our Balsamic Chicken Breast with Roasted Veggies recipe is ready in under 30 minutes and packed with flavor! Try it tonight! Recipe link in bio! #quickdinner #healthyrecipe #balsamicchicken

**Post 3:**
It's time to get your salad game on! Our Quinoa Salad with Grilled Chicken, Feta, and Honey Mustard Vinaigrette is a tasty way to sneak in some extra protein and fiber. Recipe link in bio! #saladlover #healthyrecipe #quinoasalad

**Post 4:**
Sweet treats don't have to be unhealthy! Our No-Bake Energy Bites recipe uses rolled oats, peanut butter, and honey to create bite-sized snacks that 

In [14]:
import os
import json
from pathlib import Path
from collections import defaultdict
from datetime import datetime

def create_folder_structure_json(root_path, max_depth=None, exclude_dirs=None):
    """
    Erstellt eine strukturierte JSON-Darstellung eines Ordners für LLM-Analyse
    
    Args:
        root_path: Zielverzeichnis
        max_depth: Maximale Tiefe
        exclude_dirs: Zu ignorierende Verzeichnisse
    
    Returns:
        dict mit Struktur und Metadaten
    """
    if exclude_dirs is None:
        exclude_dirs = {'.git', '__pycache__', '.DS_Store', 'node_modules', '.venv', '.idea', 'venv'}
    
    root_path = os.path.expanduser(root_path)
    
    if not os.path.exists(root_path):
        return None, f"Pfad nicht gefunden: {root_path}"
    
    structure = {
        'name': os.path.basename(root_path) or root_path,
        'type': 'folder',
        'path': root_path,
        'children': []
    }
    
    stats = {
        'total_files': 0,
        'total_folders': 0,
        'file_types': defaultdict(int),
        'depth': 0,
        'total_size': 0
    }
    
    def build_tree(path, current_depth=0, parent_struct=None):
        if max_depth and current_depth >= max_depth:
            return
        
        if current_depth > stats['depth']:
            stats['depth'] = current_depth
        
        try:
            entries = sorted(os.listdir(path))
        except PermissionError:
            return
        
        for entry in entries:
            if entry.startswith('.') and entry not in {'.gitignore', '.env', '.env.local'}:
                continue
            
            full_path = os.path.join(path, entry)
            
            try:
                if os.path.isdir(full_path):
                    if entry not in exclude_dirs:
                        stats['total_folders'] += 1
                        folder_node = {
                            'name': entry,
                            'type': 'folder',
                            'path': full_path,
                            'children': []
                        }
                        parent_struct['children'].append(folder_node)
                        build_tree(full_path, current_depth + 1, folder_node)
                else:
                    stats['total_files'] += 1
                    size = os.path.getsize(full_path)
                    stats['total_size'] += size
                    
                    ext = os.path.splitext(entry)[1] or 'no_extension'
                    stats['file_types'][ext] += 1
                    
                    file_node = {
                        'name': entry,
                        'type': 'file',
                        'extension': ext,
                        'size_bytes': size,
                        'size_mb': round(size / (1024**2), 2)
                    }
                    parent_struct['children'].append(file_node)
            except Exception as e:
                print(f"Fehler bei {full_path}: {e}")
    
    build_tree(root_path, 0, structure)
    
    return structure, stats


TARGET_PATH = "/Users/florianhaglsperger/Desktop/Wirtschaft Projekt" 
MAX_DEPTH = 4

print(f"📁 Scanne Ordnerstruktur: {TARGET_PATH}\n")

folder_structure, stats = create_folder_structure_json(TARGET_PATH, max_depth=MAX_DEPTH)

llm_input = {
    'folder_name': folder_structure['name'],
    'root_path': TARGET_PATH,
    'analysis_date': datetime.now().isoformat(),
    'statistics': {
        'total_files': stats['total_files'],
        'total_folders': stats['total_folders'],
        'nesting_depth': stats['depth'],
        'total_size_mb': round(stats['total_size'] / (1024**2), 2),
        'file_types': dict(sorted(stats['file_types'].items(), key=lambda x: x[1], reverse=True))
    },
    'structure': folder_structure
}

json_filename = f"{os.path.basename(TARGET_PATH)}_structure.json"
json_path = os.path.expanduser(f"~/{json_filename}")

with open(json_path, 'w') as f:
    json.dump(llm_input, f, indent=2)
    json_saved_path = f"JSON gespeichert unter: {json_path}"
    print(json_saved_path)

📁 Scanne Ordnerstruktur: /Users/florianhaglsperger/Desktop/Wirtschaft Projekt

JSON gespeichert unter: /Users/florianhaglsperger/Wirtschaft Projekt_structure.json


In [ ]:
import ollama

restructuring_prompt = f"""Analysiere die folgende Ordnerstruktur und gib konkrete Vorschläge zur Verbesserung:

STRUKTUR-DATEN:
{json.dumps(llm_input, indent=2)[:2000]}  # Erste 2000 Zeichen

AUFGABE:
1. Identifiziere redundante oder schlecht organisierte Ordner
2. Schlage eine bessere Hierarchie vor
3. Erkenne verwandte Dateien, die zusammengefasst werden sollten
4. Gib konkrete Umzugs-Schritte an (z.B. "Verschiebe alle .csv Dateien nach 'data/'")
5. Erkenne nicht verwendete oder archivarische Ordner

Antworte im JSON-Format mit Struktur-Vorschlägen."""

print("🤖 Sende Struktur-Analyse an Modell...\n")

response = ollama.chat(
    model='llama3',
    messages=[
        {
            'role': 'system',
            'content': 'Du bist ein Dateisystem-Experte. Analysiere Ordnerstrukturen und gib praktische Restrukturierungs-Empfehlungen in deutscher Sprache.'
        },
        {
            'role': 'user',
            'content': restructuring_prompt
        }
    ]
)

recommendations = response.message.content

print("=" * 70)
print("📋 RESTRUKTURIERUNGS-EMPFEHLUNGEN")
print("=" * 70)
print(recommendations)

# Empfehlungen speichern
recommendations_path = os.path.expanduser(f"~/{os.path.basename(TARGET_PATH)}_recommendations.txt")
with open(recommendations_path, 'w') as f:
    f.write(f"ORDNERSTRUKTUR-ANALYSE EMPFEHLUNGEN\n")
    f.write(f"Datum: {datetime.now().isoformat()}\n")
    f.write(f"Ordner: {TARGET_PATH}\n")
    f.write("=" * 70 + "\n\n")
    f.write(recommendations)

print(f"\n✅ Empfehlungen gespeichert: {recommendations_path}")

🤖 Sende Struktur-Analyse an Modell...

📋 RESTRUKTURIERUNGS-EMPFEHLUNGEN
Based on the provided data, I'll analyze the folder structure and provide recommendations for improvement.

**Recommendaions:**

1. **Redundant or poorly organized folders:** None found.
2. **Improved hierarchy:** The current structure seems logical, but considering the presence of only three files with the same extension (.pages), it might be beneficial to create a subfolder for these files. This would keep the root folder clean and allow for easier organization of similar files in the future.

**Structural suggestions:**

```
{
  "Wirtschaft Projekt": {
    "Businessplan-Vorlage-Textteil.pages",
    "Wirtschaft Businessplan.pages",
    "Wirtschaft Projektdokumentation.pages"
  }
}
```

3. **Related files to group:** None found, as all files have the same extension and seem to be related to the project.

**Grouping suggestions:**

None

4. **Concrete relocation steps:**

* Move all `.pages` files into a subfolder 